# Deploy do Modelo CatBoost de Precificação de Imóveis
## Teste rápido do modelo em produção

A célula abaixo testa o modelo sem a execução do restante do notebook: ela chama o endpoint publicado por uma requisição HTTP com token de acesso e devolve o preço previsto para um imóvel de exemplo (CEP 98103). O código roda dentro do Databricks ou em qualquer ambiente Python, como o VSCode, e outros imóveis podem ser testados alterando os valores do dicionário `imovel`.

A resposta leva cerca de 1 segundo. Com o endpoint parado (scale-to-zero), a primeira chamada pode levar alguns minutos ou exceder o tempo limite.

Por segurança, o token de acesso não é publicado no repositório e é fornecido separadamente para a avaliação.

O restante do notebook documenta como o modelo foi colocado em produção, e a mesma chamada é detalhada na seção final.

In [0]:
import requests

TOKEN = "<TOKEN_DE_ACESSO>"
URL = "https://dbc-62eb1bf2-3ada.cloud.databricks.com/serving-endpoints/modelo-precificacao-imoveis/invocations"

imovel = {"bedrooms": 3, "bathrooms": 2.5, "sqft_living": 2120, "sqft_lot": 6000, "floors": 1.5,
          "waterfront": 0, "view": 0, "condition": 4, "grade": 8, "sqft_above": 1680, "sqft_basement": 440,
          "yr_built": 1995, "yr_renovated": 0, "lat": 47.669, "long": -122.345,
          "sqft_living15": 2000, "sqft_lot15": 6000, "zipcode": 98103}

resposta = requests.post(URL, headers={"Authorization": f"Bearer {TOKEN}"},
                         json={"dataframe_records": [imovel]}, timeout=300)
resposta.raise_for_status()
print(f"Preço previsto: US$ {resposta.json()['predictions'][0]['preco_previsto']:,.0f}")

Preço previsto: US$ 741,701



Neste notebook é feito o deploy do modelo CatBoost treinado no notebook `Treinamento_e_engenharia_reversa`, que prevê o preço de imóveis (Seattle). O `modelo_final.joblib` e o `metadados.json` são carregados do Volume do Unity Catalog, e o deploy evolui em duas iterações: primeiro com a API recebendo as 46 features prontas e depois com o **join demográfico por CEP dentro do próprio modelo**, o que reduz a entrada para 17 variáveis do imóvel + `zipcode`. O modelo é registrado no Unity Catalog, publicado e testado em um endpoint de Model Serving e, no fim, validado no holdout: as previsões do modelo em produção são idênticas às da modelagem (diferença máxima de 2e-16, só arredondamento numérico).

**Métricas de teste do modelo** (do `metadados.json`): MAPE ≈ 11,5% · R² ≈ 0,909 · MAE ≈ US$ 61.901 · RMSE ≈ US$ 118.358.

## Visão geral do deploy

O deploy é organizado em duas iterações e termina com a validação no holdout:

1. **Carregamento** do `modelo_final.joblib` e do `metadados.json` do Volume `/Volumes/workspace/case_madeinweb/modelo_final/Modelo final`
2. **Validação da ordem das features:** o CatBoost foi treinado com features posicionais, então a ordem do `metadados.json` é confirmada na prática
3. **Deploy v1:** wrapper pyfunc que recebe as 46 features prontas (linha de base)
4. **Evolução com join por CEP dentro do modelo:** a entrada cai para as 17 variáveis do imóvel + `zipcode`, e o wrapper faz o enriquecimento demográfico e calcula as variáveis derivadas (é a versão servida no endpoint)
5. **Previsão do holdout** (4.295 imóveis de teste) com o modelo em produção, comparada às previsões da modelagem
6. **Teste manual do endpoint, chamada externa (como um site faria) e conclusões**

> ⚠️ A implantação do endpoint leva cerca de **15 minutos**. Cada execução das células de registro cria uma **nova versão** do modelo no Unity Catalog (comportamento normal do MLflow), e o endpoint passa a servir a última versão registrada.

In [0]:
%pip install catboost joblib
# Reinicia o processo Python para garantir que o catboost (pacote com binários) fique disponível nas células seguintes
try:
    dbutils.library.restartPython()  # noqa: F821
except Exception:
    pass

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.7/96.7 MB 87.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 120.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [catboost]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Carregando o modelo e os metadados

O modelo final (regressor CatBoost de `log_price`) e os metadados do treinamento foram salvos no Volume do Unity Catalog ao fim da etapa de modelagem. Aqui os dois são carregados, com a conferência de hiperparâmetros, métricas e número de features.

In [0]:
import json

import catboost
import joblib
import numpy as np
import pandas as pd

CAMINHO_MODELO = "/Volumes/workspace/case_madeinweb/modelo_final/Modelo final/modelo_final.joblib"
CAMINHO_META = "/Volumes/workspace/case_madeinweb/modelo_final/Modelo final/metadados.json"

with open(CAMINHO_META, "r", encoding="utf-8") as f:
    metadados = json.load(f)

modelo_catboost = joblib.load(CAMINHO_MODELO)

variaveis = metadados["variaveis"]

print(f"Modelo: {metadados['modelo']} | Variação: {metadados['variacao']} | Alvo: {metadados['alvo']}")
print(
    f"Hiperparâmetros: lr={metadados['lr']}, "
    f"n_estimators={metadados['n_estimators']}, max_depth={metadados['max_depth']}"
)
print(f"Features nos metadados: {len(variaveis)} | Features no modelo: {len(modelo_catboost.feature_names_)}")
nomes_modelo = list(modelo_catboost.feature_names_)
if set(nomes_modelo) <= {str(i) for i in range(len(nomes_modelo))}:
    # O modelo foi treinado com features posicionais ('0'..'45'): a ordem canônica
    # das colunas é a do metadados.json, validada na célula seguinte contra a base de treino.
    assert len(nomes_modelo) == len(variaveis) == 46
    print("Modelo treinado com 46 features posicionais; a ordem de `variaveis` será validada na célula seguinte.")
else:
    somente_nos_metadados = set(variaveis) - set(nomes_modelo)
    somente_no_modelo = set(nomes_modelo) - set(variaveis)
    assert not somente_nos_metadados and not somente_no_modelo, (
        f"Divergência! Só nos metadados: {somente_nos_metadados} | Só no modelo: {somente_no_modelo}"
    )
    print("Mesmas 46 features nos metadados e no modelo.")

print("\nMétricas registradas no metadados.json:")
for k, v in metadados["metricas"].items():
    if "MAPE" in k or "R2" in k:
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v:,.2f}")

print(f"\ncatboost {catboost.__version__} | joblib {joblib.__version__}")

Modelo: CatBoost | Variação: v0_base | Alvo: log_price
Hiperparâmetros: lr=0.05, n_estimators=620, max_depth=8
Features nos metadados: 46 | Features no modelo: 46
Modelo treinado com 46 features posicionais; a ordem de `variaveis` será validada na célula seguinte.

Métricas registradas no metadados.json:
  treino_MAPE: 0.0927
  cv_MAPE: 0.1134
  teste_MAPE: 0.1150
  teste_MAE: 61,901.01
  teste_RMSE: 118,358.47
  teste_R2: 0.9086

catboost 1.2.10 | joblib 1.5.2


## 2. Validação da ordem das features

O CatBoost foi treinado com features posicionais (`'0'..'45'`), então o deploy só funciona se a ordem de `variaveis` no `metadados.json` for a mesma da matriz de treino. Para confirmar, a base de treino é reconstruída (`kc_house_treino` + variáveis derivadas), a previsão é feita com as colunas nessa ordem e o MAPE é comparado com o registrado. Como contraprova, o teste é repetido com as colunas embaralhadas: se a ordem estivesse errada, o MAPE explodiria.

In [0]:
# Reconstrói a base de treino a partir da tabela limpa do Unity Catalog
base = spark.table("workspace.case_madeinweb.kc_house_treino").toPandas()
base.columns = [c.split(" - ")[0].strip() for c in base.columns]  # remove sufixo traduzido

# Variáveis derivadas (mesma lógica da etapa de modelagem, confirmada pelas faixas de treino)
if "house_age" not in base.columns:
    base["house_age"] = 2015 - base["yr_built"]
if "was_renovated" not in base.columns:
    base["was_renovated"] = (base["yr_renovated"] > 0).astype(float)
if "yr_last_update" not in base.columns:
    base["yr_last_update"] = base["yr_renovated"].where(base["yr_renovated"] > 0, base["yr_built"])

# Percentuais censitários, caso não venham prontos na tabela
percentuais = {
    "per_urbn": "urbn_ppltn_qty", "per_sbrbn": "sbrbn_ppltn_qty", "per_farm": "farm_ppltn_qty",
    "per_non_farm": "non_farm_qty", "per_less_than_9": "edctn_less_than_9_qty",
    "per_9_to_12": "edctn_9_12_qty", "per_hsd": "edctn_high_schl_qty",
    "per_some_clg": "edctn_some_clg_qty", "per_assoc": "edctn_assoc_dgre_qty",
    "per_bchlr": "edctn_bchlr_dgre_qty", "per_prfsnl": "edctn_prfsnl_qty",
}
for destino, origem in percentuais.items():
    if destino not in base.columns:
        base[destino] = 100 * base[origem] / base["ppltn_qty"]

X = base[variaveis].astype("float64").to_numpy()
preco_real = base["price"].to_numpy(dtype="float64")

mape_ordenado = np.mean(np.abs(np.exp(modelo_catboost.predict(X)) - preco_real) / preco_real)

rng = np.random.default_rng(42)
permutacao = rng.permutation(len(variaveis))
mape_embaralhado = np.mean(
    np.abs(np.exp(modelo_catboost.predict(X[:, permutacao])) - preco_real) / preco_real
)

print(f"MAPE com a ordem de `variaveis`: {mape_ordenado:.4f} ({mape_ordenado:.1%})")
print(f"MAPE com colunas embaralhadas:  {mape_embaralhado:.4f} ({mape_embaralhado:.1%})")
assert mape_ordenado < 0.15, "A ordem de `variaveis` não reproduz o desempenho do modelo!"
assert mape_ordenado < mape_embaralhado / 2, "A ordem das features parece não importar "
print("✅ Ordem das features validada: MAPE na base completa consistente com os metadados.")

MAPE com a ordem de `variaveis`: 0.0972 (9.7%)
MAPE com colunas embaralhadas:  0.3770 (37.7%)
✅ Ordem das features validada: MAPE na base completa consistente com os metadados.


## 3. Deploy v1: modelo com input das 46 features

Na primeira iteração, o endpoint recebe as **46 features prontas**: quem chama a API precisa montar o vetor completo, inclusive a demografia do CEP. É a linha de base, simplificada na seção 4.

O teste usa dois imóveis fictícios, um **padrão** e um **premium**. As variáveis da casa são definidas manualmente (valores plausíveis e coerentes entre si), e as variáveis censitárias recebem o **ponto médio** das faixas de treino, o que garante que as 46 features fiquem dentro do intervalo visto no treino.

In [0]:
faixas = metadados["faixas_treino"]

# Ponto médio de cada faixa de treino como base garantidamente dentro do intervalo
ponto_medio = {v: (faixas[v][0] + faixas[v][1]) / 2 for v in variaveis}

imovel_padrao = dict(ponto_medio)
imovel_padrao.update({
    "bedrooms": 3, "bathrooms": 2.5, "sqft_living": 2120, "sqft_lot": 6000,
    "floors": 1.5, "waterfront": 0, "view": 0, "condition": 4, "grade": 8,
    "sqft_above": 1680, "sqft_basement": 440, "yr_built": 1995, "yr_renovated": 0,
    "lat": 47.61, "long": -122.33, "sqft_living15": 2000, "sqft_lot15": 6000,
    "house_age": 20, "was_renovated": 0, "yr_last_update": 1995,
})

imovel_premium = dict(ponto_medio)
imovel_premium.update({
    "bedrooms": 4, "bathrooms": 3.5, "sqft_living": 4500, "sqft_lot": 12000,
    "floors": 2.0, "waterfront": 1, "view": 4, "condition": 4, "grade": 11,
    "sqft_above": 3500, "sqft_basement": 1000, "yr_built": 2005, "yr_renovated": 2014,
    "lat": 47.63, "long": -122.22, "sqft_living15": 4200, "sqft_lot15": 10000,
    "house_age": 10, "was_renovated": 1, "yr_last_update": 2014,
})

exemplo_df = pd.DataFrame([imovel_padrao, imovel_premium])[variaveis]

# Garantia: todos os valores dentro das faixas de treino
for coluna in variaveis:
    lo, hi = faixas[coluna]
    assert exemplo_df[coluna].between(lo, hi).all(), f"{coluna} está fora da faixa de treino!"

print("✅ Todos os 46 campos dos dois imóveis estão dentro das faixas de treino.")
display(exemplo_df)

# Sanidade: previsão direta com o CatBoost carregado
# (o modelo foi treinado com features posicionais, por isso passamos os valores via numpy)
log_pred = modelo_catboost.predict(exemplo_df.to_numpy(dtype="float64"))
for nome, lp in zip(["Imóvel padrão", "Imóvel premium"], log_pred):
    print(f"{nome}: log_price = {lp:.3f} → preço previsto ≈ US$ {np.exp(lp):,.0f}")

✅ Todos os 46 campos dos dois imóveis estão dentro das faixas de treino.


bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15,house_age,was_renovated,yr_last_update,ppltn_qty,urbn_ppltn_qty,sbrbn_ppltn_qty,farm_ppltn_qty,non_farm_qty,medn_hshld_incm_amt,medn_incm_per_prsn_amt,hous_val_amt,edctn_less_than_9_qty,edctn_9_12_qty,edctn_high_schl_qty,edctn_some_clg_qty,edctn_assoc_dgre_qty,edctn_bchlr_dgre_qty,edctn_prfsnl_qty,per_urbn,per_sbrbn,per_farm,per_non_farm,per_less_than_9,per_9_to_12,per_hsd,per_some_clg,per_assoc,per_bchlr,per_prfsnl
3,2.5,2120,6000,1.5,0,0,4,8,1680,440,1995,0,47.61,-122.33,2000,6000,20,0,1995,33609.0,32090.5,5588.0,219.5,4986.5,82375.0,49506.5,465900.0,1837.5,2267.0,6200.5,7019.5,2034.5,7349.0,4538.5,50.0,40.5,1.0,49.0,5.5,6.0,15.0,22.5,4.5,22.5,13.0
4,3.5,4500,12000,2.0,1,4,4,11,3500,1000,2005,2014,47.63,-122.22,4200,10000,10,1,2014,33609.0,32090.5,5588.0,219.5,4986.5,82375.0,49506.5,465900.0,1837.5,2267.0,6200.5,7019.5,2034.5,7349.0,4538.5,50.0,40.5,1.0,49.0,5.5,6.0,15.0,22.5,4.5,22.5,13.0


Imóvel padrão: log_price = 13.442 → preço previsto ≈ US$ 688,103
Imóvel premium: log_price = 14.675 → preço previsto ≈ US$ 2,361,227


### 3.1 Wrapper pyfunc e log no MLflow

O wrapper `PrecificadorImoveis` recebe um DataFrame com as 46 colunas, coloca as colunas na ordem que o CatBoost espera, prevê o `log_price` e devolve duas colunas: `log_price_previsto` (bruto) e `preco_previsto` (`exp(log_price)`, em dólares).

O run do MLflow registra os hiperparâmetros como **parâmetros** e os valores do `metadados.json` como **métricas e tags**.

In [0]:
import mlflow
from mlflow.models import infer_signature


class PrecificadorImoveis(mlflow.pyfunc.PythonModel):
    """Recebe um DataFrame com as 46 features do imóvel e devolve
    o log_price previsto e o preço previsto em dólares."""

    def __init__(self, colunas):
        # O modelo foi treinado com features posicionais ('0'..'45'); a ordem canônica
        # das 46 colunas vem do metadados.json e é fixada aqui no wrapper.
        self.colunas = list(colunas)

    def load_context(self, context):
        import joblib

        self.modelo = joblib.load(context.artifacts["modelo_catboost"])

    def predict(self, context, model_input):
        df = pd.DataFrame(model_input).reindex(columns=self.colunas)
        log_price = self.modelo.predict(df.to_numpy(dtype="float64"))
        return pd.DataFrame({
            "log_price_previsto": log_price,
            "preco_previsto": np.exp(log_price),
        })


# Assinatura inferida a partir dos exemplos de entrada e da saída real do modelo
saida_exemplo = pd.DataFrame({
    "log_price_previsto": log_pred,
    "preco_previsto": np.exp(log_pred),
})
assinatura = infer_signature(exemplo_df, saida_exemplo)

with mlflow.start_run() as run:
    run_id = run.info.run_id

    # Parâmetros e métricas registrados a partir do metadados.json
    mlflow.log_params({
        "variacao": metadados["variacao"],
        "alvo": metadados["alvo"],
        "tipo_modelo": metadados["modelo"],
        "lr": metadados["lr"],
        "n_estimators": metadados["n_estimators"],
        "max_depth": metadados["max_depth"],
        "n_features": len(variaveis),
    })
    for metrica, valor in metadados["metricas"].items():
        mlflow.log_metric(metrica, valor)
    mlflow.set_tags({
        "projeto": "case_madeinweb",
        "tarefa": "precificacao_imoveis",
        "cidade": "Seattle",
    })

    model_info = mlflow.pyfunc.log_model(
        name="modelo",
        python_model=PrecificadorImoveis(variaveis),
        artifacts={"modelo_catboost": CAMINHO_MODELO},
        input_example=exemplo_df.head(2),
        signature=assinatura,
        pip_requirements=[f"catboost=={catboost.__version__}", f"joblib=={joblib.__version__}"],
    )

print("Run do MLflow:", run_id)
print("URI do artefato:", model_info.model_uri)

/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://ww

Run do MLflow: b7019318577949ad93da4d772d6e1fcc
URI do artefato: models:/m-dedc87bf0f214b3fa3b364e249446f0b


### 3.2 Validação do artefato em ambiente isolado

Antes do registro, o artefato é testado com `mlflow.models.predict` e `env_manager="uv"`, que reproduz o que o servidor fará: cria um ambiente virtual limpo, instala os `pip_requirements` **do zero** e faz uma previsão. O modelo só é registrado se essa validação passar.

In [0]:
import tempfile

# mlflow.models.predict escreve a previsão em um arquivo (output_path) e não retorna valor
with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as tmp:
    caminho_saida = tmp.name

mlflow.models.predict(
    model_uri=model_info.model_uri,
    input_data=exemplo_df.head(2),
    env_manager="uv",
    output_path=caminho_saida,
)

with open(caminho_saida, "r", encoding="utf-8") as f:
    conteudo = json.load(f)

if isinstance(conteudo, dict) and "predictions" in conteudo:
    conteudo = conteudo["predictions"]
saida_validacao = pd.DataFrame(conteudo)

display(saida_validacao)
assert saida_validacao.shape == (2, 2), f"Formato inesperado: {saida_validacao.shape}"
assert saida_validacao["preco_previsto"].between(10000, 10000000).all()
print("✅ Artefato validado em ambiente isolado: dependências instaláveis e previsão OK.")

2026/11/25 18:33:36 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2026/11/25 18:33:37 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs/mlflow-35524a5e395988ca0af2747939e4c74b306ae31e with python version 3.12.3 using uv
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/virtualenv_envs/mlflow-35524a5e395988ca0af2747939e4c74b306ae31e
Activate with: source /tmp/virtualenv_envs/mlflow-35524a5e395988ca0af2747939e4c74b306ae31e/bin/activate
2026/11/25 18:33:38 INFO mlflow.utils.virtualenv: Installing dependencies
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-35524a5e395988ca0af2747939e4c74b306ae31e
Resolved 3 packages in 224ms
 Downloaded pip
 Downloaded setuptools
Prepared 3 packages in 187ms
Installed 3 packages in 69ms
 + pip==25.3
 + setuptools==80.9.0
 + wheel==0.45.1
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-35524a5e395988ca0af2747939e4c74b306ae31e
Resolved 92 packages in 1.12s
 Downloaded kiwisolver
 Downloaded databricks-sdk
 Downloade

log_price_previsto,preco_previsto
13.441694014122069,688103.1370079143
14.674691777717074,2361226.576325435


✅ Artefato validado em ambiente isolado: dependências instaláveis e previsão OK.


### 3.3 Registro no Unity Catalog

O modelo validado é registrado em `workspace.case_madeinweb.modelo_precificacao_imoveis` (Unity Catalog), o que centraliza versionamento, governança e serving.

In [0]:
NOME_MODELO_REGISTRADO = "workspace.case_madeinweb.modelo_precificacao_imoveis"

mlflow.set_registry_uri("databricks-uc")
versao_registrada = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=NOME_MODELO_REGISTRADO,
    await_registration_for=300,
)
model_version = versao_registrada.version

print(f"✅ Modelo registrado: {NOME_MODELO_REGISTRADO} | versão {model_version}")
print(f"URI: models:/{NOME_MODELO_REGISTRADO}/{model_version}")

Registered model 'workspace.case_madeinweb.modelo_precificacao_imoveis' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Modelo registrado: workspace.case_madeinweb.modelo_precificacao_imoveis | versão 4
URI: models:/workspace.case_madeinweb.modelo_precificacao_imoveis/4


🔗 Created version '4' of model 'workspace.case_madeinweb.modelo_precificacao_imoveis': https://dbc-62eb1bf2-3ada.cloud.databricks.com/explore/data/models/workspace/case_madeinweb/modelo_precificacao_imoveis/version/4?o=7474648164256438


### 3.4 Endpoint de Model Serving

A versão registrada é publicada no endpoint `modelo-precificacao-imoveis` com workload **CPU Small** (o modelo tem só 2,6 MB) e **scale-to-zero** habilitado, para evitar custo de ociosidade. A contrapartida é o *cold start*: a primeira chamada depois de um período parado demora mais. A implantação leva cerca de 15 minutos, e a célula seguinte acompanha o status até ficar READY.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)

NOME_ENDPOINT = "modelo-precificacao-imoveis"

w = WorkspaceClient()

# Se o endpoint já existe servindo exatamente este modelo, atualiza; caso contrário, cria
try:
    endpoint_atual = w.serving_endpoints.get(NOME_ENDPOINT)
except NotFound:
    endpoint_atual = None

entidade_servida = ServedEntityInput(
    entity_name=NOME_MODELO_REGISTRADO,
    entity_version=str(model_version),
    workload_type=ServingModelWorkloadType("CPU"),  # modelo leve (2,6 MB) → CPU
    workload_size="Small",
    scale_to_zero_enabled=True,
)
config_endpoint = EndpointCoreConfigInput(name=NOME_ENDPOINT, served_entities=[entidade_servida])

if endpoint_atual is None:
    w.serving_endpoints.create(name=NOME_ENDPOINT, config=config_endpoint)
    print(f"Endpoint '{NOME_ENDPOINT}' criado — implantação em andamento (~15 min).")
else:
    entidades = endpoint_atual.config.served_entities if endpoint_atual.config else []
    nomes = [e.entity_name for e in entidades]
    if len(entidades) != 1 or nomes[0] != NOME_MODELO_REGISTRADO:
        raise ValueError(
            f"O endpoint '{NOME_ENDPOINT}' serve {nomes} e não será substituído. "
            "Escolha outro nome ou confirme a substituição."
        )
    w.serving_endpoints.update_config(name=NOME_ENDPOINT, served_entities=[entidade_servida])
    print(f"Endpoint '{NOME_ENDPOINT}' atualizado para a versão {model_version}.")

Endpoint 'modelo-precificacao-imoveis' atualizado para a versão 4.


In [0]:
# Aguarda o endpoint ficar READY (polling com status a cada 15 s)
import time

from databricks.sdk.service.serving import EndpointStateConfigUpdate, EndpointStateReady


def aguardar_endpoint_pronto(nome, timeout_s=900, intervalo_s=15):
    inicio = time.time()
    estados_falha = {
        EndpointStateConfigUpdate.UPDATE_FAILED,
        EndpointStateConfigUpdate.UPDATE_CANCELED,
    }
    while time.time() - inicio < timeout_s:
        estado = w.serving_endpoints.get(nome).state
        if (
            estado.ready == EndpointStateReady.READY
            and estado.config_update == EndpointStateConfigUpdate.NOT_UPDATING
        ):
            print(f"✅ Endpoint '{nome}' READY em {int(time.time() - inicio)}s")
            return
        if estado.config_update in estados_falha:
            raise RuntimeError(f"Deploy falhou com estado {estado.config_update.value}")
        print(
            f"{int(time.time() - inicio)}s: ready={estado.ready}, "
            f"config_update={estado.config_update}"
        )
        time.sleep(intervalo_s)
    raise TimeoutError(f"Endpoint '{nome}' não ficou READY em {timeout_s}s")


aguardar_endpoint_pronto(NOME_ENDPOINT)

0s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
15s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
30s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
45s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
60s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
75s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
90s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
105s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
✅ Endpoint 'modelo-precificacao-imoveis' READY em 120s


### 3.5 Teste do endpoint (v1)

O endpoint é chamado com os dois imóveis de exemplo (todas as features dentro das faixas de treino) e o preço previsto é exibido. Ao final, a célula imprime um exemplo de chamada via `curl`, em que `<TOKEN>` representa um token de acesso pessoal (Settings → Developer → Access tokens).

In [0]:
import json

resposta = w.serving_endpoints.query(
    name=NOME_ENDPOINT,
    dataframe_records=exemplo_df.to_dict(orient="records"),
)

for nome, previsao in zip(["Imóvel padrão", "Imóvel premium"], resposta.predictions):
    print(
        f"{nome}: log_price = {previsao['log_price_previsto']:.3f} | "
        f"preço previsto ≈ US$ {previsao['preco_previsto']:,.0f}"
    )

# Exemplo de chamada via curl (<TOKEN> = token de acesso pessoal)
payload = {"dataframe_records": [imovel_padrao]}
print("\n--- Exemplo de chamada via curl ---")
print(
    "curl -X POST \\\n"
    f"  {w.config.host}/serving-endpoints/{NOME_ENDPOINT}/invocations \\\n"
    '  -H "Authorization: Bearer <SEU_TOKEN>" \\\n'
    '  -H "Content-Type: application/json" \\\n'
    f"  -d '{json.dumps(payload)}'"
)

Imóvel padrão: log_price = 13.442 | preço previsto ≈ US$ 688,103
Imóvel premium: log_price = 14.675 | preço previsto ≈ US$ 2,361,227

--- Exemplo de chamada via curl ---
curl -X POST \
  https://dbc-62eb1bf2-3ada.cloud.databricks.com/serving-endpoints/modelo-precificacao-imoveis/invocations \
  -H "Authorization: Bearer <SEU_TOKEN>" \
  -H "Content-Type: application/json" \
  -d '{"dataframe_records": [{"bedrooms": 3, "bathrooms": 2.5, "sqft_living": 2120, "sqft_lot": 6000, "floors": 1.5, "waterfront": 0, "view": 0, "condition": 4, "grade": 8, "sqft_above": 1680, "sqft_basement": 440, "yr_built": 1995, "yr_renovated": 0, "lat": 47.61, "long": -122.33, "sqft_living15": 2000, "sqft_lot15": 6000, "house_age": 20, "was_renovated": 0, "yr_last_update": 1995, "ppltn_qty": 33609.0, "urbn_ppltn_qty": 32090.5, "sbrbn_ppltn_qty": 5588.0, "farm_ppltn_qty": 219.5, "non_farm_qty": 4986.5, "medn_hshld_incm_amt": 82375.0, "medn_incm_per_prsn_amt": 49506.5, "hous_val_amt": 465900.0, "edctn_less_than_9

## 4. Evolução: join por CEP dentro do modelo

A v1 obriga quem chama a API a montar as 46 features, o que é pouco prático e abre espaço para erro. Nesta iteração, o **enriquecimento passa para dentro do wrapper**:

* A entrada cai para as **17 variáveis do imóvel + `zipcode`**
* O wrapper faz o **join com a tabela demográfica do CEP** (26 variáveis). A tabela vai embutida no artefato como CSV, porque o container de serving não acessa tabelas do Unity Catalog
* O wrapper **calcula** `house_age`, `was_renovated` e `yr_last_update` a partir de `yr_built` e `yr_renovated`, com a mesma lógica do treino
* O CatBoost recebe as 46 features na **ordem validada na seção 2**, sem risco de variáveis fora de ordem

Cada execução da célula seguinte registra uma **nova versão** do modelo no Unity Catalog, e o endpoint passa a servir a última. Antes do registro, o artefato é validado de novo em ambiente isolado (`uv`), com as dependências instaladas do zero.

### 4.1 Wrapper com join por CEP, validação e registro

O wrapper `HousePriceModel` é testado na sessão, validado em ambiente isolado (`uv`) e registrado como nova versão no Unity Catalog. A versão registrada fica em `model_version_v2`, usada nas células seguintes.

In [0]:
import json
import tempfile

import catboost
import joblib
import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature

BASE = "/Volumes/workspace/case_madeinweb/modelo_final/Modelo final"
MODELO = f"{BASE}/modelo_final.joblib"
META = f"{BASE}/metadados.json"
DEMO = f"{BASE}/zipcode_demographics.csv"
NOME_UC = "workspace.case_madeinweb.modelo_precificacao_imoveis"

# Exporta a tabela de CEP do Unity Catalog para CSV no volume.
# O container de serving não acessa tabelas do Unity Catalog — o CSV viaja DENTRO do artefato.
spark.table("workspace.case_madeinweb.zipcode_demographics").toPandas().to_csv(DEMO, index=False)

with open(META) as f:
    meta = json.load(f)
demo = pd.read_csv(DEMO)

# ---- Destinação das 46 variáveis do modelo ----
variaveis_demo = [c for c in demo.columns if c != "zipcode"]      # 26: vêm do join com a tabela de CEP
variaveis_derivadas = ["house_age", "was_renovated", "yr_last_update"]  # 3: calculadas pelo wrapper
variaveis_casa = [v for v in meta["variaveis"] if v not in variaveis_demo + variaveis_derivadas]  # 17: input da API
assert len(variaveis_demo) + len(variaveis_derivadas) + len(variaveis_casa) == len(meta["variaveis"]) == 46


class HousePriceModel(mlflow.pyfunc.PythonModel):
    """Recebe as variáveis do imóvel + zipcode, enriquece com a demografia
    do CEP (join), deriva as variáveis calculadas e devolve o preço previsto."""

    def load_context(self, context):
        self.model = joblib.load(context.artifacts["modelo"])
        with open(context.artifacts["metadados"]) as f:
            self.vars = json.load(f)["variaveis"]
        self.demo = pd.read_csv(context.artifacts["demografia"])

    def predict(self, context, model_input):
        df = pd.DataFrame(model_input).copy()

        # CEPs informados precisam existir na tabela demográfica
        faltantes = set(df["zipcode"]) - set(self.demo["zipcode"])
        if faltantes:
            raise ValueError(f"CEPs sem dados demográficos: {sorted(faltantes)}")

        # 1) Enriquecimento: join com a tabela de CEP (26 variáveis demográficas)
        df = df.merge(self.demo, on="zipcode", how="left")

        # 2) Variáveis derivadas (mesma lógica do treino, ano de referência 2015)
        df["house_age"] = 2015 - df["yr_built"]
        df["was_renovated"] = (df["yr_renovated"] > 0).astype(float)
        df["yr_last_update"] = df["yr_renovated"].where(df["yr_renovated"] > 0, df["yr_built"])

        # 3) As 46 variáveis na ordem canônica do metadados.json (via numpy: modelo posicional)
        X = df[self.vars].astype("float64").to_numpy()
        log_pred = self.model.predict(X)
        return pd.DataFrame({"preco_previsto": np.exp(log_pred), "log_price_previsto": log_pred})


artifacts = {"modelo": MODELO, "metadados": META, "demografia": DEMO}

# Exemplo de entrada realista: as 17 variáveis do imóvel + zipcode
# (casa típica de Seattle no CEP 98103, com lat/long coerentes com o CEP)
imovel_exemplo = {
    "bedrooms": 3, "bathrooms": 2.5, "sqft_living": 2120, "sqft_lot": 6000,
    "floors": 1.5, "waterfront": 0, "view": 0, "condition": 4, "grade": 8,
    "sqft_above": 1680, "sqft_basement": 440, "yr_built": 1995, "yr_renovated": 0,
    "lat": 47.669, "long": -122.345, "sqft_living15": 2000, "sqft_lot15": 6000,
    "zipcode": 98103,
}
assert set(imovel_exemplo) == set(variaveis_casa) | {"zipcode"}
assert 98103 in set(demo["zipcode"]), "CEP 98103 sem dados demográficos!"
exemplo = pd.DataFrame([imovel_exemplo])

mlflow.set_registry_uri("databricks-uc")

class _Ctx:
    artifacts = artifacts


modelo_local = HousePriceModel()
modelo_local.load_context(_Ctx())
previsao_exemplo = modelo_local.predict(None, exemplo)
display(previsao_exemplo)
signature = infer_signature(exemplo, previsao_exemplo)

# 1) Log do artefato (requirements com versão real do catboost)
with mlflow.start_run(run_name="join_por_cep_interno") as run:
    model_info_v2 = mlflow.pyfunc.log_model(
        name="model",
        python_model=HousePriceModel(),
        artifacts=artifacts,
        signature=signature,
        input_example=exemplo,
        pip_requirements=[f"catboost=={catboost.__version__}", f"joblib=={joblib.__version__}"],
    )

# 2) Validação em ambiente isolado (uv): instala as dependências do zero e prevê
with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as tmp:
    caminho_saida = tmp.name
mlflow.models.predict(
    model_uri=model_info_v2.model_uri,
    input_data=exemplo,
    env_manager="uv",
    output_path=caminho_saida,
)
with open(caminho_saida, "r", encoding="utf-8") as f:
    conteudo = json.load(f)
previsao_isolada = pd.DataFrame(conteudo["predictions"] if isinstance(conteudo, dict) else conteudo)
assert previsao_isolada["preco_previsto"].iloc[0] > 0
assert np.isclose(previsao_isolada["preco_previsto"].iloc[0], previsao_exemplo["preco_previsto"].iloc[0], rtol=1e-6)
print("✅ Artefato validado em ambiente isolado (uv): join por CEP e previsão OK.")

# 3) Registro como nova versão no Unity Catalog
versao_v2 = mlflow.register_model(
    model_uri=model_info_v2.model_uri,
    name=NOME_UC,
    await_registration_for=300,
)
model_version_v2 = versao_v2.version
print(f"✅ Nova versão registrada: {NOME_UC} | versão {model_version_v2} (join por zipcode dentro do modelo)")

/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


preco_previsto,log_price_previsto
741700.5789020733,13.516700908231387


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-62eb1bf2-3ada.cloud.databricks.com/ml/experiments/4254828317153571/models/m-14c7b2337bde4d2d801a97c314f6e1ac?o=7474648164256438
2026/11/25 20:41:45 INFO mlflow.pyfunc: Validating input example against mod

2026/11/25 20:41:51 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2026/11/25 20:41:52 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs/mlflow-35524a5e395988ca0af2747939e4c74b306ae31e with python version 3.12.3 using uv
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/virtualenv_envs/mlflow-35524a5e395988ca0af2747939e4c74b306ae31e
Activate with: source /tmp/virtualenv_envs/mlflow-35524a5e395988ca0af2747939e4c74b306ae31e/bin/activate
2026/11/25 20:41:53 INFO mlflow.utils.virtualenv: Installing dependencies
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-35524a5e395988ca0af2747939e4c74b306ae31e
Resolved 3 packages in 174ms
 Downloaded setuptools
 Downloaded pip
Prepared 3 packages in 154ms
Installed 3 packages in 65ms
 + pip==25.3
 + setuptools==80.9.0
 + wheel==0.45.1
Using Python 3.12.3 environment at: /tmp/virtualenv_envs/mlflow-35524a5e395988ca0af2747939e4c74b306ae31e
Resolved 92 packages in 1.18s
 Downloaded pydantic-core
 Downloaded aiohttp
 Downloaded ki

✅ Artefato validado em ambiente isolado (uv): join por CEP e previsão OK.


Registered model 'workspace.case_madeinweb.modelo_precificacao_imoveis' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/15 [00:00<?, ?it/s]

🔗 Created version '7' of model 'workspace.case_madeinweb.modelo_precificacao_imoveis': https://dbc-62eb1bf2-3ada.cloud.databricks.com/explore/data/models/workspace/case_madeinweb/modelo_precificacao_imoveis/version/7?o=7474648164256438


✅ Nova versão registrada: workspace.case_madeinweb.modelo_precificacao_imoveis | versão 7 (join por zipcode dentro do modelo)


### 4.2 Atualização e teste do endpoint

O endpoint passa a servir a versão registrada na 4.1 e é testado com o imóvel de exemplo (17 variáveis + CEP).

In [0]:
# 4.2) Atualiza o endpoint para a versão com join por CEP e testa
import json
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointStateConfigUpdate,
    EndpointStateReady,
    ServedEntityInput,
    ServingModelWorkloadType,
)

NOME_ENDPOINT = "modelo-precificacao-imoveis"
w = WorkspaceClient()

entidade_v2 = ServedEntityInput(
    entity_name=NOME_UC,
    entity_version=str(model_version_v2),
    workload_type=ServingModelWorkloadType("CPU"),
    workload_size="Small",
    scale_to_zero_enabled=True,
)
w.serving_endpoints.update_config(
    name=NOME_ENDPOINT,
    served_entities=[entidade_v2],
)
print(f"Endpoint '{NOME_ENDPOINT}' atualizado para a versão {model_version_v2}. Aguardando READY...")


def aguardar_endpoint_pronto(nome, timeout_s=900, intervalo_s=15):
    inicio = time.time()
    estados_falha = {
        EndpointStateConfigUpdate.UPDATE_FAILED,
        EndpointStateConfigUpdate.UPDATE_CANCELED,
    }
    while time.time() - inicio < timeout_s:
        estado = w.serving_endpoints.get(nome).state
        if (
            estado.ready == EndpointStateReady.READY
            and estado.config_update == EndpointStateConfigUpdate.NOT_UPDATING
        ):
            print(f"✅ Endpoint '{nome}' READY em {int(time.time() - inicio)}s")
            return
        if estado.config_update in estados_falha:
            raise RuntimeError(f"Deploy falhou com estado {estado.config_update.value}")
        print(
            f"{int(time.time() - inicio)}s: ready={estado.ready}, "
            f"config_update={estado.config_update}"
        )
        time.sleep(intervalo_s)
    raise TimeoutError(f"Endpoint '{nome}' não ficou READY em {timeout_s}s")


aguardar_endpoint_pronto(NOME_ENDPOINT)

# Teste com as 17 variáveis do imóvel + zipcode (demografia vem do join interno)
resposta_v2 = w.serving_endpoints.query(
    name=NOME_ENDPOINT,
    dataframe_records=exemplo.to_dict(orient="records"),
)
previsao = resposta_v2.predictions[0]
print(f"CEP {int(exemplo['zipcode'].iloc[0])}: preço previsto ≈ US$ {previsao['preco_previsto']:,.0f}")
assert previsao["preco_previsto"] > 0

# Exemplo de chamada via curl (bem mais simples: só o imóvel + CEP!)
payload_v2 = {"dataframe_records": exemplo.to_dict(orient="records")}
print("\n--- Exemplo de chamada via curl (substitua <SEU_TOKEN>) ---")
print(
    "curl -X POST \\\n"
    f"  {w.config.host}/serving-endpoints/{NOME_ENDPOINT}/invocations \\\n"
    '  -H "Authorization: Bearer <SEU_TOKEN>" \\\n'
    '  -H "Content-Type: application/json" \\\n'
    f"  -d '{json.dumps(payload_v2)}'"
)

Endpoint 'modelo-precificacao-imoveis' atualizado para a versão 5. Aguardando READY...
0s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
15s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
30s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
45s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
60s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
75s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
90s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
105s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
120s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
135s: ready=EndpointStateReady.READY, config_update=EndpointStateConfigUpdate.IN_PROGRESS
✅ Endpoint 'modelo-pr

## 5. Previsão do holdout com o modelo deployado

Para comprovar que o modelo em produção é o mesmo do treinamento, a versão registrada na seção 4 (a mesma servida no endpoint) prevê a tabela `workspace.case_madeinweb.holdout_previsoes`, com os 4.295 imóveis de teste. A entrada é só as variáveis do imóvel + `zipcode`: o join com o CEP e as variáveis derivadas acontecem dentro do wrapper. O resultado é comparado com o preço real e com a coluna `previsto` salva na modelagem, e gravado em `holdout_previsoes_deploy`.

In [0]:
import json

import mlflow
import numpy as np
import pandas as pd

BASE = "/Volumes/workspace/case_madeinweb/modelo_final/Modelo final"
NOME_UC = "workspace.case_madeinweb.modelo_precificacao_imoveis"

# 1) Carrega o modelo registrado (mesmo artefato servido no endpoint)
try:
    model_version_v2  # noqa: B018 — definido na célula 4.1
except NameError as e:
        raise RuntimeError("model_version_v2 não definido: a célula 4.1 (log e registro do modelo) precisa ser executada antes") from e
modelo_deployado = mlflow.pyfunc.load_model(f"models:/{NOME_UC}/{model_version_v2}")

# 2) Holdout: input = 17 variáveis do imóvel + zipcode, nos tipos da assinatura do modelo
tipos = {
    col.name: ("float64" if col.type.name == "double" else "int64")
    for col in modelo_deployado.metadata.get_input_schema().inputs
}

demo = pd.read_csv(f"{BASE}/zipcode_demographics.csv")
holdout = spark.table("workspace.case_madeinweb.holdout_previsoes").toPandas()
print(f"Holdout: {len(holdout)} imóveis")

# CEPs do holdout precisam existir na tabela demográfica
faltantes = set(holdout["zipcode"]) - set(demo["zipcode"])
assert not faltantes, f"CEPs sem demografia: {sorted(faltantes)}"

X_holdout = holdout[list(tipos)].astype(tipos)

# 3) Previsão com o modelo deployado (join por CEP + variáveis derivadas dentro do wrapper)
previsao = modelo_deployado.predict(X_holdout)

resultado = holdout[["id", "zipcode", "price", "previsto", "erro_pct"]].copy()
resultado["preco_previsto_deploy"] = previsao["preco_previsto"].to_numpy()
resultado["erro_pct_deploy"] = (
    (resultado["preco_previsto_deploy"] - resultado["price"]) / resultado["price"]
)

# 4) Comparação com o real e com a previsão original da modelagem
mape_deploy = resultado["erro_pct_deploy"].abs().mean()
mape_original = resultado["erro_pct"].abs().mean()
dif_max = (
    (resultado["preco_previsto_deploy"] - resultado["previsto"]).abs()
    / resultado["previsto"]
).max()
print(f"MAPE do modelo deployado: {mape_deploy:.4f} ({mape_deploy:.1%})")
print(f"MAPE da modelagem original (coluna previsto): {mape_original:.4f} ({mape_original:.1%})")
print(f"Diferença máxima deploy vs. previsto original: {dif_max:.2e}")
display(resultado.head(5))

# 5) Salva o resultado numa tabela própria (não altera a tabela original)
spark.createDataFrame(resultado).write.mode("overwrite").saveAsTable(
    "workspace.case_madeinweb.holdout_previsoes_deploy"
)
print("✅ Resultado salvo em workspace.case_madeinweb.holdout_previsoes_deploy")

Holdout: 4295 imóveis
MAPE do modelo deployado: 0.1150 (11.5%)
MAPE da modelagem original (coluna previsto): 0.1150 (11.5%)
Diferença máxima deploy vs. previsto original: 2.08e-16


id,zipcode,price,previsto,erro_pct,preco_previsto_deploy,erro_pct_deploy
5200087,98108,487000.0,523968.1189331366,0.07590989513991087,523968.1189331366,0.07590989513991084
7200080,98055,239000.0,260231.62425485544,0.08883524792826547,260231.62425485544,0.08883524792826546
7400062,98118,299800.0,252672.78275316203,-0.15719552117024005,252672.78275316203,-0.15719552117024008
11900140,98042,254000.0,296757.7491466903,0.16833759506570978,296757.7491466903,0.16833759506570986
49000051,98146,350000.0,324861.2423467071,-0.07182502186655115,324861.2423467071,-0.07182502186655117


✅ Resultado salvo em workspace.case_madeinweb.holdout_previsoes_deploy


## 5.1 Previsão dos imóveis novos (`future_unseen_examples`)

O arquivo `future_unseen_examples.csv` traz 100 imóveis sem preço, exatamente no formato de entrada da API: as 17 variáveis do imóvel + `zipcode`. O modelo registrado (o mesmo servido no endpoint) faz o join com o CEP, calcula as variáveis derivadas e prevê o preço de cada um, como aconteceria em produção.

Cada imóvel recebe também um **alerta de faixa**: a coluna `fora_da_faixa` lista as variáveis com valor fora do intervalo visto no treino (vazia quando está tudo dentro). Nesses casos, a previsão deve ser lida como referência, porque modelos de árvore não extrapolam além do que viram. Nenhum dos 100 imóveis ficou fora da faixa.

Como verificação, as previsões são comparadas com uma execução independente fora do Databricks, em que o `modelo_final.joblib` é carregado diretamente e recebe o mesmo join por CEP e as mesmas variáveis derivadas (`unseen_previsoes_local.csv`). A diferença máxima foi de 2,09e-16, só arredondamento numérico. O resultado é salvo na tabela `unseen_previsoes` e em `unseen_previsoes.csv`.

In [0]:
import json

import mlflow
import pandas as pd

BASE = "/Volumes/workspace/case_madeinweb/modelo_final/Modelo final"
NOME_UC = "workspace.case_madeinweb.modelo_precificacao_imoveis"

try:
    model_version_v2  # noqa: B018 — definido na célula 4.1
except NameError as e:
    raise RuntimeError("Rode primeiro a célula 4.1 (log/registro do modelo com join por CEP)") from e
modelo_deployado = mlflow.pyfunc.load_model(f"models:/{NOME_UC}/{model_version_v2}")

# Entrada nos tipos da assinatura do modelo (17 variáveis do imóvel + zipcode)
tipos = {
    col.name: ("float64" if col.type.name == "double" else "int64")
    for col in modelo_deployado.metadata.get_input_schema().inputs
}
unseen = pd.read_csv(f"{BASE}/future_unseen_examples.csv")
previsao = modelo_deployado.predict(unseen[list(tipos)].astype(tipos))

# Alerta de faixa: variáveis do imóvel fora do intervalo visto no treino
with open(f"{BASE}/metadados.json", encoding="utf-8") as f:
    faixas = json.load(f)["faixas_treino"]
fora = [
    ", ".join(c for c in tipos if c in faixas and not faixas[c][0] <= linha[c] <= faixas[c][1])
    for _, linha in unseen.iterrows()
]

resultado = unseen.assign(preco_previsto=previsao["preco_previsto"].to_numpy(), fora_da_faixa=fora)

# Comparação com a execução independente fora do Databricks (modelo carregado do joblib)
local = pd.read_csv(f"{BASE}/unseen_previsoes_local.csv")
dif_max = ((resultado["preco_previsto"] - local["preco_previsto"]).abs() / local["preco_previsto"]).max()
print(f"{len(resultado)} imóveis previstos | diferença máxima deploy vs. local: {dif_max:.2e}")
print(f"Imóveis com alguma variável fora da faixa do treino: {(resultado['fora_da_faixa'] != '').sum()}")
display(resultado)

spark.createDataFrame(resultado).write.mode("overwrite").saveAsTable("workspace.case_madeinweb.unseen_previsoes")
resultado.to_csv(f"{BASE}/unseen_previsoes.csv", index=False)
print("✅ Salvo em workspace.case_madeinweb.unseen_previsoes e unseen_previsoes.csv")

100 imóveis previstos | diferença máxima deploy vs. local: 2.09e-16
Imóveis com alguma variável fora da faixa do treino: 0


bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,preco_previsto,fora_da_faixa
4,1.0,1680,5043,1.5,0,0,4,6,1680,0,1911,0,98118,47.5354,-122.273,1560,5765,356615.76838766,
3,2.5,2220,6380,1.5,0,0,4,8,1660,560,1931,0,98115,47.6974,-122.313,950,6380,594440.1406709591,
3,2.25,1630,10962,1.0,0,0,4,8,1100,530,1977,0,98030,47.3801,-122.166,1830,8470,277200.42421762395,
5,2.5,1710,9720,2.0,0,0,4,8,1710,0,1974,0,98005,47.5903,-122.157,2270,9672,614821.0789007804,
2,1.0,850,6370,1.0,0,0,3,6,850,0,1951,0,98126,47.5198,-122.373,850,5170,226918.38923168334,
4,2.75,2630,4501,2.0,0,0,3,8,2630,0,2015,0,98028,47.7748,-122.244,2380,4599,537641.0072593195,
4,1.75,2290,7380,1.0,0,0,3,7,1390,900,1963,0,98178,47.5034,-122.245,1170,7381,297912.1039229007,
4,2.0,1730,9139,2.0,0,0,3,8,1730,0,1957,0,98125,47.7181,-122.316,1410,7311,445356.10023757425,
3,1.75,2190,8500,1.0,0,0,4,9,2190,0,1957,0,98112,47.64,-122.285,2850,8868,1244001.8317717256,
3,1.75,1850,22767,1.5,0,4,5,6,1850,0,1908,0,98074,47.6144,-122.067,2700,17906,633639.4828135319,


✅ Salvo em workspace.case_madeinweb.unseen_previsoes e unseen_previsoes.csv


## 5.2 Teste manual do endpoint

Os dados do imóvel são informados nos campos (widgets) do topo do notebook, e a célula abaixo envia esses valores ao endpoint. A entrada é só as 17 variáveis da casa + CEP, e a demografia do bairro vem do join interno. A primeira chamada depois de um período parado pode demorar por causa do scale-to-zero.

In [0]:
from databricks.sdk import WorkspaceClient

NOME_ENDPOINT = "modelo-precificacao-imoveis"

# campo: (rótulo, valor padrão) — padrão = casa típica do CEP 98103
campos = {
    "zipcode": ("CEP", "98103"),
    "bedrooms": ("Quartos", "3"),
    "bathrooms": ("Banheiros (0.5 = lavabo)", "2.5"),
    "sqft_living": ("Área útil (pés²)", "2120"),
    "sqft_lot": ("Terreno (pés²)", "6000"),
    "floors": ("Andares", "1.5"),
    "waterfront": ("Frente para a água (0 ou 1)", "0"),
    "view": ("Vista (0 a 4)", "0"),
    "condition": ("Conservação (1 a 5)", "4"),
    "grade": ("Padrão construtivo (1 a 13)", "8"),
    "sqft_above": ("Área acima do solo (pés²)", "1680"),
    "sqft_basement": ("Área do porão (pés²)", "440"),
    "yr_built": ("Ano de construção", "1995"),
    "yr_renovated": ("Ano da reforma (0 = nunca)", "0"),
    "lat": ("Latitude", "47.669"),
    "long": ("Longitude", "-122.345"),
    "sqft_living15": ("Área útil média dos 15 vizinhos (pés²)", "2000"),
    "sqft_lot15": ("Terreno médio dos 15 vizinhos (pés²)", "6000"),
}
for campo, (rotulo, padrao) in campos.items():
    dbutils.widgets.text(campo, padrao, rotulo)

# Campos com decimais vão como float; os demais como inteiro (tipos da assinatura do modelo)
decimais = {"bathrooms", "floors", "lat", "long"}
imovel = {
    c: float(dbutils.widgets.get(c)) if c in decimais else int(float(dbutils.widgets.get(c)))
    for c in campos
}

resposta = WorkspaceClient().serving_endpoints.query(name=NOME_ENDPOINT, dataframe_records=[imovel])
print(f"Preço previsto: US$ {resposta.predictions[0]['preco_previsto']:,.0f}")

Preço previsto: US$ 741,701


## 5.3 Chamada externa ao endpoint (como um site ou aplicativo faria)

A célula anterior chama o endpoint de dentro do Databricks, pelo `WorkspaceClient`. Esta célula faz a mesma chamada de fora da plataforma, por uma requisição HTTP comum (`requests.post`) com token de acesso, que é o caminho que um site, aplicativo ou sistema do time comercial usaria em produção. O código roda em qualquer ambiente Python, como o VSCode, sem depender do Databricks.

A entrada é a mesma: as 17 variáveis do imóvel + CEP, com a demografia do bairro vinda do join interno do modelo. O resultado é idêntico ao da chamada pelo `WorkspaceClient` e ao do teste da seção 4.

**Latência:** a resposta levou cerca de 1 segundo, um tempo adequado para uso em sites e aplicativos, onde a previsão aparece para o usuário logo após o preenchimento dos dados. A exceção é a primeira chamada depois de um período parado, que demora mais por causa do scale-to-zero (*cold start*).

> ⚠️ O token de acesso não é publicado no repositório: ele é temporário, fornecido separadamente para a avaliação e revogado ao fim do processo. Em produção, o token fica em um cofre de segredos (Databricks Secrets, Azure Key Vault ou variável de ambiente), nunca no código.

In [0]:
import requests

TOKEN = "<SOLICITE ACESSO AO TOKEN PARA TESTAR>"
URL = "https://dbc-62eb1bf2-3ada.cloud.databricks.com/serving-endpoints/modelo-precificacao-imoveis/invocations"

imovel = {"bedrooms": 3, "bathrooms": 2.5, "sqft_living": 2120, "sqft_lot": 6000, "floors": 1.5,
          "waterfront": 0, "view": 0, "condition": 4, "grade": 8, "sqft_above": 1680, "sqft_basement": 440,
          "yr_built": 1995, "yr_renovated": 0, "lat": 47.669, "long": -122.345,
          "sqft_living15": 2000, "sqft_lot15": 6000, "zipcode": 98103}

resposta = requests.post(URL, headers={"Authorization": f"Bearer {TOKEN}"},
                         json={"dataframe_records": [imovel]}, timeout=300)
resposta.raise_for_status()
print(f"Preço previsto: US$ {resposta.json()['predictions'][0]['preco_previsto']:,.0f}")

Preço previsto: US$ 741,701


## 6. Conclusões

**Resultado:** o modelo CatBoost de precificação de imóveis de King County (Seattle) (`v0_base`: lr = 0,05, 620 estimadores, depth = 8, MAPE de teste 11,5%, R² 0,909) está em produção como API REST no Databricks Model Serving, com o **join demográfico por CEP feito dentro do próprio modelo**.

**Artefatos entregues:**

| Item | Valor |
| --- | --- |
| Modelo registrado (Unity Catalog) | `workspace.case_madeinweb.modelo_precificacao_imoveis` |
| Endpoint (CPU Small, scale-to-zero) | `modelo-precificacao-imoveis` |
| Contrato da API | 17 variáveis do imóvel + `zipcode` → `preco_previsto` e `log_price_previsto` |
| Previsões do holdout | `workspace.case_madeinweb.holdout_previsoes_deploy` |
| Previsões dos imóveis novos | `workspace.case_madeinweb.unseen_previsoes` |

**Garantias de qualidade ao longo do notebook:**

1. **Ordem das features validada empiricamente**: MAPE de 9,7% com a ordem do `metadados.json` contra 37,7% com colunas embaralhadas (o modelo foi treinado com features posicionais)
2. **Artefato validado em ambiente isolado** (`uv`): dependências instaláveis do zero, como o serving fará
3. **Holdout de 4.295 imóveis reproduzido**: MAPE de 11,5%, com previsões idênticas às da modelagem original até a 15ª casa decimal (diferença máxima de 2e-16): o modelo em produção é exatamente o treinado
4. **Robustez do wrapper**: CEPs sem demografia são rejeitados com erro claro; variáveis derivadas calculadas internamente, eliminando o risco de input inconsistente
5. **Imóveis novos previstos de forma consistente**: os 100 imóveis do `future_unseen_examples` foram previstos pelo modelo registrado e conferidos contra uma execução independente fora do Databricks (diferença máxima de 2,09e-16), todos dentro das faixas de treino

**Exemplo de consumo** (`<TOKEN>` = token de acesso pessoal, gerado em *Settings → Developer → Access tokens*):

```bash
curl -X POST \
  https://dbc-62eb1bf2-3ada.cloud.databricks.com/serving-endpoints/modelo-precificacao-imoveis/invocations \
  -H "Authorization: Bearer <TOKEN>" \
  -H "Content-Type: application/json" \
  -d '{"dataframe_records": [{"bedrooms": 3, "bathrooms": 2.5, "sqft_living": 2120, "sqft_lot": 6000, "floors": 1.5, "waterfront": 0, "view": 0, "condition": 4, "grade": 8, "sqft_above": 1680, "sqft_basement": 440, "yr_built": 1995, "yr_renovated": 0, "lat": 47.669, "long": -122.345, "sqft_living15": 2000, "sqft_lot15": 6000, "zipcode": 98103}]}'
```